In [43]:
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.over_sampling import ADASYN
from imblearn.over_sampling import RandomOverSampler

In [7]:
def discretize(df):
    df["hba1c_category"] = pd.cut(
            df['HbA1c_level'],
            bins=[0, 5.7, 6.5, float("inf")],
            labels=["Normal", "Prediabetes", "Diabetes"],
            right= False
        )
    
    df['bmi_category'] = pd.cut(
            df['bmi'],
            bins=[0, 18.5, 25, 30, float("inf")],
            labels=["Underweight", "Healthy Weight", "Overweight", "Obesity"],
            right= False
        )
    
    df['blood_glucose_category'] = pd.cut(
            df['blood_glucose_level'],
            bins=[70, 140, 200, float("inf")],
            labels=["Normal", "Prediabetes", "Possible Diabetes"],
            right= False
        )
    
    return df

def add_engineered_features(df):
    # Boolean and Logical Combinations
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.6).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)
    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

In [25]:
train_df = pd.read_csv("../../data/split/train.csv") 
validation_df = pd.read_csv("../../data/split/validation.csv") 

train_df = add_engineered_features(discretize(train_df))
validation_df =  add_engineered_features(discretize(validation_df))

train_df.head(5)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,hba1c_category,bmi_category,blood_glucose_category,high_hba1c_flag,senior_flag,cardio_risk_flag
0,Female,49.0,0,0,No Info,23.21,5.8,100,0,Prediabetes,Healthy Weight,Normal,0,0,0
1,Female,59.0,0,0,former,32.60,6.5,140,0,Diabetes,Obesity,Prediabetes,0,0,0
2,Male,32.0,0,0,former,25.84,5.8,155,0,Prediabetes,Overweight,Prediabetes,0,0,0
3,Female,70.0,0,0,ever,36.90,6.5,155,0,Diabetes,Obesity,Prediabetes,0,1,0
4,Male,55.0,0,0,never,24.77,3.5,200,0,Normal,Healthy Weight,Possible Diabetes,0,0,0


In [26]:
cols_to_drop = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

flag_cols = ["hypertension", "heart_disease", "high_hba1c_flag", "senior_flag", "cardio_risk_flag"]


train_cat_df = train_df.drop(columns=cols_to_drop, axis=1)
train_cat_df[flag_cols] = train_cat_df[flag_cols].replace({1: "Yes", 0: "No"})

val_cat_df = validation_df.drop(columns=cols_to_drop, axis=1)
val_cat_df[flag_cols] = val_cat_df[flag_cols].replace({1: "Yes", 0: "No"})


train_cat_df.head(5)

,gender,hypertension,heart_disease,smoking_history,diabetes,hba1c_category,bmi_category,blood_glucose_category,high_hba1c_flag,senior_flag,cardio_risk_flag
0,Female,No,No,No Info,0,Prediabetes,Healthy Weight,Normal,No,No,No
1,Female,No,No,former,0,Diabetes,Obesity,Prediabetes,No,No,No
2,Male,No,No,former,0,Prediabetes,Overweight,Prediabetes,No,No,No
3,Female,No,No,ever,0,Diabetes,Obesity,Prediabetes,No,Yes,No
4,Male,No,No,never,0,Normal,Healthy Weight,Possible Diabetes,No,No,No


In [30]:
X_train = train_cat_df.drop("diabetes", axis=1)
y_train = train_cat_df["diabetes"]

X_test = val_cat_df.drop("diabetes", axis=1)
y_test = val_cat_df["diabetes"]


model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=X_train.columns.tolist(),
    eval_metric="Recall",
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

preds = model.predict(X_test)
preds_proba = model.predict_proba(X_test)[:, 1]
print("=" * 40)
print("         MODEL EVALUATION METRICS")
print("=" * 40)
print(f"Accuracy   : {accuracy_score(y_test, preds):.4f}")
print(f"Precision  : {precision_score(y_test, preds):.4f}")
print(f"Recall     : {recall_score(y_test, preds):.4f}")
print(f"F1 Score   : {f1_score(y_test, preds):.4f}")
print(f"ROC-AUC    : {roc_auc_score(y_test, preds_proba):.4f}")
print("=" * 40)
print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=["No Diabetes", "Diabetes"]))


0:	learn: 0.1895216	test: 0.1988994	best: 0.1988994 (0)	total: 47.7ms	remaining: 23.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.4858490566
bestIteration = 13

Shrink model to first 14 iterations.
         MODEL EVALUATION METRICS
Accuracy   : 0.9395
Precision  : 0.7410
Recall     : 0.4858
F1 Score   : 0.5869
ROC-AUC    : 0.9348

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.95      0.98      0.97     13115
    Diabetes       0.74      0.49      0.59      1272

    accuracy                           0.94     14387
   macro avg       0.85      0.73      0.78     14387
weighted avg       0.93      0.94      0.93     14387



In [31]:
# add scale weights for class imbalance

X_train = train_cat_df.drop("diabetes", axis=1)
y_train = train_cat_df["diabetes"]

X_test = val_cat_df.drop("diabetes", axis=1)
y_test = val_cat_df["diabetes"]


model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=X_train.columns.tolist(),
    auto_class_weights="Balanced",  # CatBoost handles it automatically
    eval_metric="F1",
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

preds = model.predict(X_test)
preds_proba = model.predict_proba(X_test)[:, 1]
print("=" * 40)
print("         MODEL EVALUATION METRICS")
print("=" * 40)
print(f"Accuracy   : {accuracy_score(y_test, preds):.4f}")
print(f"Precision  : {precision_score(y_test, preds):.4f}")
print(f"Recall     : {recall_score(y_test, preds):.4f}")
print(f"F1 Score   : {f1_score(y_test, preds):.4f}")
print(f"ROC-AUC    : {roc_auc_score(y_test, preds_proba):.4f}")
print("=" * 40)
print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=["No Diabetes", "Diabetes"]))


0:	learn: 0.8508810	test: 0.8460321	best: 0.8460321 (0)	total: 63.2ms	remaining: 31.6s
100:	learn: 0.8678039	test: 0.8639743	best: 0.8649641 (71)	total: 6.06s	remaining: 23.9s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8649641093
bestIteration = 71

Shrink model to first 72 iterations.
         MODEL EVALUATION METRICS
Accuracy   : 0.8451
Precision  : 0.3507
Recall     : 0.8829
F1 Score   : 0.5020
ROC-AUC    : 0.9432

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.99      0.84      0.91     13115
    Diabetes       0.35      0.88      0.50      1272

    accuracy                           0.85     14387
   macro avg       0.67      0.86      0.71     14387
weighted avg       0.93      0.85      0.87     14387



In [ ]:
"""
Yes, Recall went from 0.48 → 0.88 which is great, but Precision dropped to 0.35 — meaning a lot of false alarms (healthy people flagged as diabetic).
The model swung too far the other way. You need to balance the tradeoff by tuning the classification threshold instead of defaulting to 0.5
"""

# Find the best threshold using F1 or Recall target
from sklearn.metrics import precision_recall_curve
import numpy as np

preds_proba = model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, preds_proba)

# Option 1 — maximize F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best F1 Threshold: {best_threshold:.4f}")

# Option 2 — minimum recall threshold (0.8)
min_recall = 0.8
valid = recalls[:-1] >= min_recall
best_threshold = thresholds[valid][np.argmax(precisions[:-1][valid])]
print(f"Best Threshold for Recall >= 0.8: {best_threshold:.4f}")

# Apply chosen threshold
preds_custom = (preds_proba >= best_threshold).astype(int)

print(f"\nAt threshold {best_threshold:.4f}:")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1 Score  : {f1_score(y_test, preds_custom):.4f}")

Best F1 Threshold: 0.7869
Best Threshold for Recall >= 0.8: 0.6286

At threshold 0.6286:
Precision : 0.4561
Recall    : 0.8011
F1 Score  : 0.5813


In [44]:
X_train = train_cat_df.drop("diabetes", axis=1)
y_train = train_cat_df["diabetes"]

X_test = val_cat_df.drop("diabetes", axis=1)
y_test = val_cat_df["diabetes"]

print("\n ================ RandomOverSampler ================== \n")
print("Before:", y_train.value_counts().to_dict())

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print("After :", y_train_res.value_counts().to_dict())

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=X_train.columns.tolist(),
    auto_class_weights="Balanced",  # CatBoost handles it automatically
    eval_metric="F1",
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train_res, y_train_res, eval_set=(X_test, y_test))

preds = model.predict(X_test)
preds_proba = model.predict_proba(X_test)[:, 1]
print("=" * 40)
print("         MODEL EVALUATION METRICS")
print("=" * 40)
print(f"Accuracy   : {accuracy_score(y_test, preds):.4f}")
print(f"Precision  : {precision_score(y_test, preds):.4f}")
print(f"Recall     : {recall_score(y_test, preds):.4f}")
print(f"F1 Score   : {f1_score(y_test, preds):.4f}")
print(f"ROC-AUC    : {roc_auc_score(y_test, preds_proba):.4f}")
print("=" * 40)
print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=["No Diabetes", "Diabetes"]))



 ================ RandomOverSampler ================== 

Before: {0: 61203, 1: 5936}
After : {0: 61203, 1: 61203}
0:	learn: 0.8512269	test: 0.4908359	best: 0.4908359 (0)	total: 114ms	remaining: 56.7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5056792367
bestIteration = 35

Shrink model to first 36 iterations.
         MODEL EVALUATION METRICS
Accuracy   : 0.8488
Precision  : 0.3556
Recall     : 0.8750
F1 Score   : 0.5057
ROC-AUC    : 0.9424

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.99      0.85      0.91     13115
    Diabetes       0.36      0.88      0.51      1272

    accuracy                           0.85     14387
   macro avg       0.67      0.86      0.71     14387
weighted avg       0.93      0.85      0.87     14387

